# 01 · Preprocessing, external holdout and inputs for R

**Brief sections 3, 4, 5 and 10.** This notebook turns the borehole CSV into the 0.5 m classification table, reserves the external Downtown boreholes before any modelling, and writes the inputs for the R spatial analysis.

| Step | Rule |
|---|---|
| Source | `data/raw/SiemReap_BH_Master_Downtown_DEM_Region_v1_ArcGIS.csv`, checked against its SHA-256 |
| Target | CL, CH → **Clay = 1**; SC, SM, SP → **Sand = 0**; every other symbol (ML, CL-ML, missing) is excluded, never converted |
| Depth | 0–15 m below ground, 0.5 m cells |
| Labelling | A cell is labelled from the sample intervals that overlap it (≥ 50 % coverage by one class, none of the other). Records longer than 3 m are not treated as measured strata |
| Gap filling | An unsampled gap between two measured cells of the same class is filled with that class, if no record of any kind touches it (section 2b). Filled cells are **training data only**; every test, tuning score and threshold uses measured cells. This deviates from the brief (design report section 10) |
| Predictors | `UTM_Easting_m`, `UTM_Northing_m`, `Top_Elevation_m = Ground_Elevation_m − cell top depth` |
| Traceability | Every cell keeps its `BH_Code` and the CSV line numbers it came from |
| External holdout | Chosen from coordinates and sampling metadata only (no class values), locked by a SHA-256 committed in `config/external_holdout.lock.json` |

Outputs with borehole identifiers go to `data/processed/` and `outputs/…/restricted/` (git-ignored). `outputs/…/aggregate/` holds counts only.

In [ ]:
import os, sys, json
from pathlib import Path

# Run mode: "full" for the experiment, "smoke" for the reduced software validation run.
# os.environ["SIEMREAP_MODE"] = "smoke"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "project.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from siemreap import config
cfg = config.load_config(ROOT)
MODE = config.run_mode()
print(config.smoke_banner())

from siemreap import data, geo
AGG = config.stage_dir(cfg, "01_preprocessing", "aggregate")
RES = config.stage_dir(cfg, "01_preprocessing", "restricted")
PROC = config.processed_dir(cfg)

## 1. Raw data and borehole table

In [ ]:
raw = data.load_raw(cfg)          # stops if the file differs from the configured checksum
bh = data.borehole_table(raw, cfg)
print(f"{len(raw):,} records, {raw.BH_Code.nunique()} boreholes, {raw.Project_No.nunique()} projects")
display(bh.groupby("Region").agg(boreholes=("E", "size"), projects=("Project_No", "nunique"),
                                 depth_max_median_m=("depth_max_m", "median"),
                                 ground_min_m=("Ground_Elevation_m", "min"), ground_max_m=("Ground_Elevation_m", "max")))

In [ ]:
# USCS symbols on records starting above 15 m, by region (review of symbols outside the binary map)
top = raw[raw.Depth_From_m < cfg["depth"]["bottom_m"]]
display(pd.crosstab(top.USCS.fillna("(missing)"), top.Region, margins=True))

**Symbols outside the binary map.** ML and CL-ML occur only outside Downtown (in the 5 km ring and the 10 km zone). Per the brief they are not converted to Clay; they are excluded. Missing USCS symbols are not imputed. `scripts/01_audit_raw_data.py` writes the full audit, including a lab-fines check of every logged symbol.

## 2. 0.5 m discretisation

In [ ]:
cells, info = data.discretise(raw, cfg)
print(json.dumps(info, indent=1))
assert np.allclose(cells.Top_Elevation_m, cells.Ground_Elevation_m - cells.depth_top_m)
display(cells.groupby("Region").agg(boreholes=("BH_Code", "nunique"), cells=("y", "size"), clay_share=("y", "mean")).round(3))

## 2b. Gap filling (training data only)

Samples are typically 0.45–0.5 m long and 1.5 m apart, so most of each borehole between samples is unsampled. When the measured cells directly above and below a gap have the **same class**, the gap is filled with that class. A gap is left unlabelled when:

* the classes above and below differ (the boundary lies somewhere inside it);
* any record touches it: a record of the other class, a symbol outside the binary map (ML, CL-ML), a missing symbol, or an ignored long record;
* it is longer than `fill_max_gap_m`;
* it lies above the first or below the last measured cell.

Filled cells carry `measured = 0`. They are used to **train** models only. Outer test folds, inner validation scores, thresholds, the external evaluation, the variograms and the R folds all use measured cells, so every reported score is against measured soil. Settings: `fill_gaps` and `fill_max_gap_m` in `[depth]`.

In [ ]:
cells, fill_info = data.fill_gaps(cells, raw, cfg)
print(json.dumps(fill_info, indent=1))
display(cells.assign(kind=np.where(cells.measured == 1, "measured", "filled"))
             .groupby(["Region", "kind"]).agg(boreholes=("BH_Code", "nunique"), cells=("y", "size"), clay_share=("y", "mean")).round(3))

## 3. External holdout (reserved before any modelling)

The selection procedure is fixed in `config/project.toml` (`[external]`) and implemented in `siemreap.data.select_external`. It reads coordinates, depths and the *presence* of a USCS symbol, never the class:

1. **Eligible:** Downtown boreholes reaching 15 m with ≥ 10 sampled intervals in 0–15 m, a text coordinate that agrees with the UTM fields (≤ 5 m), no record longer than 1 m, and no entry in the optional git-ignored manual-flags file.
2. **Sites:** single-linkage clusters at 100 m.
3. **Generalisation group:** whole sites whose nearest borehole outside the site is ≥ 500 m away, spread over 3 k-means zones.
4. **Interpolation group:** one borehole from each of several multi-borehole sites; its site neighbours stay in development.

On first run the SHA-256 of the sorted borehole codes is written to `config/external_holdout.lock.json`. On every later run the selection is recomputed and must match the lock, so the external set cannot drift. The external boreholes' labels are not used again until notebook 05, after the final model is frozen.

In [ ]:
flags = data.manual_flags(cfg)
external, sel_info = data.select_external(bh, cfg, flags)
lock = data.lock_external(cfg, external, sel_info)
print(json.dumps(sel_info, indent=1))
print("locked SHA-256:", lock["sha256"])
display(external.groupby("group").agg(boreholes=("site", "size"), sites=("site", "nunique"),
                                     min_isolation_m=("site_isolation_m", "min")))
data.write_csv(external, PROC / "external_holdout.csv")

## 4. Model table: development set and training extents

In [ ]:
mt = data.model_table(cells, external.index, cfg, flags)
bhs = data.boreholes_of(mt)
display(pd.crosstab(mt.set, mt.Region, values=mt.BH_Code, aggfunc="nunique", margins=True).fillna(0).astype(int))
meas = mt.measured == 1
print("Scenario A (Downtown only):", int(bhs.in_A.sum()), "boreholes,", int((mt.in_A & meas).sum()), "measured +",
      int((mt.in_A & ~meas).sum()), "filled cells")
print("Scenario B (Downtown + 5 km):", int(bhs.in_B.sum()), "boreholes,", int((mt.in_B & meas).sum()), "measured +",
      int((mt.in_B & ~meas).sum()), "filled cells")
print("Test boreholes (both scenarios): the", int(bhs.in_A.sum()), "Downtown development boreholes, via the R folds")

In [ ]:
rule_check = data.fill_rule_check(mt[mt.set == "development"])
print("Gap-filling assumption, checked on measured development cells:", rule_check)
dev = data.measured(mt[mt.set == "development"])   # descriptive statistics and R inputs: measured cells only
bands = cfg["depth"]["bands_m"]
desc = (dev.assign(band=pd.cut(dev.depth_top_m, bands, right=False))
          .groupby(["Region", "band"], observed=True).y.agg(cells="size", clay_share="mean").round(3))
display(desc)
fig, ax = plt.subplots(figsize=(4.8, 5.4), dpi=150)
from siemreap import plots
plots.clay_share_by_depth(dev, ax=ax)
ax.set_title("Development cells: clay share by 1.5 m slice", loc="left", fontsize=10)
fig.savefig(AGG / "fig_development_clay_share_by_depth.png", bbox_inches="tight")

## 5. Prediction domain (geometry only)

These are the 25 m grid columns inside the Downtown polygon, with the distance from each column to the nearest **development** Downtown borehole. R uses the columns for `blockCV::cv_distance`. The distances are the prediction-distance distribution that the validation design is compared with.

In [ ]:
poly = geo.load_polygon(cfg, "downtown_shp")
gx, gy = geo.grid_columns(poly, cfg["grid3d"]["spacing_m"])
pred_cols = pd.DataFrame({"X": gx, "Y": gy})
data.write_csv(pred_cols, PROC / "prediction_columns.csv", index=False)
dev_dt_xy = bhs[bhs.in_A][["UTM_Easting_m", "UTM_Northing_m"]].values
d = geo.nearest_distance(pred_cols.values, dev_dt_xy)
pdist = {"columns": int(len(d)), **{f"p{q}": round(float(np.percentile(d, q)), 1) for q in (5, 25, 50, 75, 90, 95)},
         "max": round(float(d.max()), 1),
         **{f"share_within_{t}m": round(float((d <= t).mean()), 4) for t in (100, 300, 500, 1000)}}
print(json.dumps(pdist, indent=1))

## 6. Write processed files and the R settings

In [ ]:
data.write_csv(mt, PROC / "cells_all.csv", index=False)
r_cols = ["BH_Code", "UTM_Easting_m", "UTM_Northing_m", "depth_top_m", "Top_Elevation_m", "y", "Region"]
dev_out = dev[r_cols].assign(is_downtown=dev.is_downtown.astype(int))
data.write_csv(dev_out, PROC / "cells_development.csv", index=False)
data.write_csv(bhs, PROC / "boreholes.csv")

r_settings = {"mode": MODE, "crs_epsg": cfg["data"]["crs_epsg"], "seed": cfg["seeds"]["global"],
              "site_linkage_m": cfg["external"]["site_linkage_m"], "blockcv": cfg["blockcv"], "spatial": cfg["spatial"],
              "files": {"cells": "data/processed/cells_development.csv",
                        "pred_points": "data/processed/prediction_columns.csv",
                        "folds_dir": cfg["paths"]["folds"]}}
(PROC / "r_settings.json").write_text(json.dumps(r_settings, indent=1))

summary = {"discretisation": info, "gap_filling": fill_info, "gap_filling_rule_check": rule_check,
           "external_selection": sel_info, "external_sha256": lock["sha256"],
           "cells_by_set": mt.set.value_counts().to_dict(), "measured_cells_by_set": mt[meas].set.value_counts().to_dict(),
           "scenario_A": {"boreholes": int(bhs.in_A.sum()), "measured_cells": int((mt.in_A & meas).sum()),
                          "filled_cells": int((mt.in_A & ~meas).sum())},
           "scenario_B": {"boreholes": int(bhs.in_B.sum()), "measured_cells": int((mt.in_B & meas).sum()),
                          "filled_cells": int((mt.in_B & ~meas).sum())},
           "prediction_distance_to_development_boreholes_m": pdist, "environment": config.environment_record()}
(AGG / "preprocessing_summary.json").write_text(json.dumps(summary, indent=1, default=str))
print("written:", [p.name for p in PROC.iterdir()])

## Next step: R (run here, or in RStudio)

The spatial-autocorrelation analysis and the blockCV folds are made in R, with the original blockCV package. The cell below runs the three R scripts from this notebook if it can find `Rscript` (on the PATH, or in `C:\Program Files\R\R-*\bin`) and shows R's messages as they come. The first run installs the R packages, which needs internet and, on Windows, Rtools. Otherwise run them in RStudio after opening `SiemReapML.Rproj`:

```
source("R/00_install_packages.R")        # once: CRAN packages + blockCV 4.0-1 from data/raw/blockCV-master
source("R/01_spatial_autocorrelation.R") # variograms, Moran's I, blockCV cv_spatial_autocor
source("R/02_blockcv_folds.R")           # blockCV folds for 100/200/300 m + diagnostics (~5-10 min)
```

Then continue with `02_spatial_folds_review.ipynb`.

In [ ]:
import glob, shutil, subprocess

RUN_R_FROM_NOTEBOOK = True
RSCRIPT = None   # set the full path if R is not found, e.g. r"C:\Program Files\R\R-4.4.1\bin\Rscript.exe"

def find_rscript():
    if RSCRIPT:
        return RSCRIPT
    found = shutil.which("Rscript")
    if found:
        return found
    candidates = sorted(glob.glob(r"C:\Program Files\R\R-*\bin\Rscript.exe"))
    return candidates[-1] if candidates else None

rscript = find_rscript()
if not RUN_R_FROM_NOTEBOOK:
    print("Skipped: run the R scripts in RStudio.")
elif rscript is None:
    print("Rscript not found. Install R (and Rtools on Windows), set RSCRIPT above, or run the scripts in RStudio.")
else:
    print("Rscript:", rscript)
    for script in ["R/00_install_packages.R", "R/01_spatial_autocorrelation.R", "R/02_blockcv_folds.R"]:
        print("\n>>", script, flush=True)
        with subprocess.Popen([rscript, script], cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding="utf-8", errors="replace") as proc:
            for line in proc.stdout:          # R messages appear as they are written
                print(line, end="", flush=True)
        if proc.returncode != 0:
            raise RuntimeError(f"{script} failed (see the messages above)")
    print("\nR steps finished: continue with 02_spatial_folds_review.ipynb")